# Sequence -> ProtT5 Embedding -> BioData Neighbors

Generate a ProtT5 embedding from a raw amino-acid sequence and query nearest neighbors from BioData.

## Notes

- This notebook uses `CBBIO.embeddings.Generator` (factory) with `model_class="protT5"` for generation.
- The generator returns **per-residue** embeddings. Pooling is intentionally done here (user-side).
- You need a running BioData PostgreSQL instance configured via `config.yaml` or `BIODATA_*` env vars.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
import numpy as np
import pandas as pd

from CBBIO.BioData import BioDataClient, NotFoundError
from CBBIO.embeddings import GenerationInput, Generator


import torch

device='cpu'
if(torch.cuda.is_available()):
    print("cuda avaible:", torch.cuda.get_device_name(0))
    device="cuda"


cuda avaible: NVIDIA GeForce RTX 5090


In [8]:
# User inputs
QUERY_ID = "MMS19_MOUSE"
QUERY_SEQUENCE = "MAAATGLEEAVAPMGALCGLVQDFVMGQQEGPADQVAADVKSGGYTVLQVVEALGSSLENAEPRTRARGAQLLSQVLLQCHSLLSEKEVVHLILFYENRLKDHHLVVPSVLQGLRALSMSVALPPGLAVSVLKAIFQEVHVQSLLQVDRHTVFSIITNFMRSREEELKGLGADFTFGFIQVMDGEKDPRNLLLAFRIVHDLISKDYSLGPFVEELFEVTSCYFPIDFTPPPNDPYGIQREDLILSLRAVLASTPRFAEFLLPLLIEKVDSEILSAKLDSLQTLNACCAVYGQKELKDFLPSLWASIRREVFQTASERVEAEGLAALHSLTACLSCSVLRADAEDLLGSFLSNILQDCRHHLCEPDMKLVWPSAKLLQAAAGASARACEHLTSNVLPLLLEQFHKHSQSNQRRTILEMILGFLKLQQKWSYEDRDERPLSSFKDQLCSLVFMALTDPSTQLQLVGIRTLTVLGAQPGLLSAEDLELAVGHLYRLTFLEEDSQSCRVAALEASGTLATLYPGAFSRHLLPKLAEELHKGESDVARADGPTKCSRHFRCLQALSAVSTHPSIVKETLPLLLQHLCQANKGNMVTESSEVVAVCQSLQQVAEKCQQDPESYWYFHKTAVPCLFALAVQASMPEKESSVLRKVLLEDEVLAALASVIGTATTHLSPELAAQSVTCIVPLFLDGNTSFLPENSFPDQFQPFQDGSSGQRRLVALLTAFVCSLPRNVEIPQLNRLMRELLKQSCGHSCPFSSTAATKCFAGLLNKQPPGQQLEEFLQLAVGTVEAGLASESSRDQAFTLLLWVTKALVLRYHPLSACLTTRLMGLLSDPELGCAAADGFSLLMSDCTDVLTRAGHADVRIMFRQRFFTDNVPALVQGFHAAPQDVKPNYLKGLSHVLNRLPKPVLLPELPTLLSLLLEALSCPDSVVQLSTLSCLQPLLLEAPQIMSLHVDTLVTKFLNLSSSYSMAVRIAALQCMHALTRLPTSVLLPYKSQVIRALAKPLDDKKRLVRKEAVSARGEWFLLGSPGS"
REQUESTED_LAYERS = [0]  # ProtT5 HF indexing: 24 is last hidden layer
POOLING = "mean"  # one of: mean, max

EMBEDDING_TYPE_NAME = "Prot-T5"  # must exist in sequence_embedding_type.name

## Generate Prot T5 embeddings

In [9]:
generator = Generator(model_class=EMBEDDING_TYPE_NAME, device=device)
result = generator.generate(
    [GenerationInput(id=QUERY_ID, sequence=QUERY_SEQUENCE)],
    layer_index=REQUESTED_LAYERS,
)

if result.errors:
    raise RuntimeError(result.errors)

print(result.model_metadata)
print(result.run_metadata)

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

T5EncoderModel LOAD REPORT from: Rostlab/prot_t5_xl_uniref50
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ModelMetadata(provider='huggingface-transformers', model_name='Rostlab/prot_t5_xl_uniref50', model_reference='Rostlab/prot_t5_xl_uniref50', model_revision='973be27c52ee6474de9c945952a8008aeb2a1a73', tokenizer_name='Rostlab/prot_t5_xl_uniref50', tokenizer_revision=None, device='cuda', framework_versions={'transformers': '5.3.0', 'torch': '2.10.0'}, parameters={'representation': 'per-residue', 'pooling': 'none', 'layer_indexing': 'biodata_reversed_0_is_last_hidden'})
RunMetadata(run_id='a6e4e23a-3ea1-4920-8e27-16b80fda94b1', created_at_utc='2026-03-11T10:23:54.785203+00:00', sequence_count=1, requested_layers=[0], resolved_layers=[0], failure_count=0, parameters={'model_reference': 'Rostlab/prot_t5_xl_uniref50'})


## Protein level pooling

In [10]:
# Pick one generated layer for DB lookup
target_layer = REQUESTED_LAYERS[0] if REQUESTED_LAYERS else result.records[0].layer_index
layer_record = next((r for r in result.records if r.layer_index == target_layer), None)
if layer_record is None:
    raise RuntimeError(f"Layer {target_layer} was not generated.")

per_residue = np.asarray(layer_record.embedding, dtype=np.float32)
if per_residue.ndim != 2:
    raise RuntimeError(f"Expected 2D per-residue embedding, got shape={per_residue.shape}")

if POOLING == "mean":
    query_embedding = per_residue.mean(axis=0)
elif POOLING == "max":
    query_embedding = per_residue.max(axis=0)
else:
    raise ValueError("POOLING must be one of: mean, max")

print({
    "layer": int(layer_record.layer_index),
    "per_residue_shape": tuple(per_residue.shape),
    "pooled_shape": tuple(query_embedding.shape),
    "sequence_len": int(len(QUERY_SEQUENCE)),
    "embedding_fingerprint": float(query_embedding[:10].sum()),
    "embedding_l2": float(np.linalg.norm(query_embedding)),
})


{'layer': 0, 'per_residue_shape': (1031, 1024), 'pooled_shape': (1024,), 'sequence_len': 1031, 'embedding_fingerprint': 0.02961084432899952, 'embedding_l2': 1.0637322664260864}


## Find neighbors in the DB

In [11]:

# BioData lookup configuration

TOP_K = 10
METRIC = "cosine"  # l2, cosine, inner_product

# Adapter convention: the protT5 generator returned by the factory uses BioData layer numbering
# (layer 0 = last hidden layer). No extra reversal is needed here.
layer_index_in_db = int(target_layer)

# create DB client and connect
client = BioDataClient()
client.connect()

emb_type = client.get_embedding_type_by_name(EMBEDDING_TYPE_NAME)
if emb_type is None:
    raise NotFoundError(
        f"Embedding type '{EMBEDDING_TYPE_NAME}' not found in BioData. "
        "Check sequence_embedding_type table or change EMBEDDING_TYPE_NAME."
    )

print({
    "embedding_type_id": emb_type.id,
    "embedding_type_name": emb_type.name,
    "generated_layer": int(target_layer),
    "db_layer_index": int(layer_index_in_db),
})

neighbors = client.find_nearest_neighbors(
    query_embedding.tolist(),
    embedding_type_id=emb_type.id,
    layer_index=layer_index_in_db,
    k=TOP_K,
    metric=METRIC,
)

neighbor_ids = [n.protein_id for n in neighbors]
annotations = client.fetch_go_annotations(neighbor_ids)

rows = []
for rank, n in enumerate(neighbors, start=1):
    rows.append({
        "rank": rank,
        "protein_id": n.protein_id,
        "distance": n.distance,
        "layer_index": n.layer_index,
        "go_terms": len(annotations.get(n.protein_id, [])),
    })

pd.DataFrame(rows)


{'embedding_type_id': 3, 'embedding_type_name': 'Prot-T5', 'generated_layer': 0, 'db_layer_index': 0}


,rank,protein_id,distance,layer_index,go_terms
0,1,MMS19_MOUSE,0.009990,0,2
1,2,MMS19_HUMAN,0.012593,0,10
2,3,MROH1_HUMAN,0.117329,0,3
3,4,H0Y746_HUMAN,0.123297,0,2
4,5,H0Y650_HUMAN,0.125256,0,4
5,6,DAAF5_MOUSE,0.127208,0,1
6,7,DAAF5_HUMAN,0.131943,0,9
7,8,DAAF5_XENLA,0.155276,0,1
8,9,TTI1_HUMAN,0.167774,0,6
9,10,MRO2B_MOUSE,0.169463,0,5


In [12]:
# Optional: inspect GO annotations for the top hit
if neighbors:
    top_id = neighbors[0].protein_id
    df=pd.DataFrame([a.__dict__ for a in annotations.get(top_id, [])])
    print("Nearest Neighbor:",top_id)
    print(df)
else:
    print("No neighbors returned.")
    
    


Nearest Neighbor: MMS19_MOUSE
        go_id category                                    description  \
0  GO:0019899        F                                 enzyme binding   
1  GO:0097361        C  cytosolic [4Fe-4S] assembly targeting complex   

  evidence_code  
0           IPI  
1           IDA  


In [ ]:
client.close()